In [ ]:
# Load Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pickle
import os  
from sklearn.model_selection import train_test_split, RepeatedKFold, RandomizedSearchCV
from sklearn.feature_selection import RFECV, RFE
from sklearn.metrics import mean_squared_error
from sklearn.ensemble import GradientBoostingRegressor  
from sklearn import set_config

# Input Data

In [ ]:
interaction_data = pd.read_csv('trainingdata/3.trainingdata_defaults.tsv', delimiter='\t') #TODO CHANGE & rerun with different sets
output_dir='4.outputs/panaroo_default/' #TODO CHANGE PATH AS NEEDED
os.makedirs(os.path.dirname(output_dir), exist_ok=True)

''' 
# Downsample for testing
interaction_small = interaction_data.sample(n=300, random_state=42)

target = 'Score'  # <-- change if needed

# keep target + a subset of features
cols_to_keep = (
    [target] + 
    interaction_data.drop(columns=[target]).sample(n=500, axis=1, random_state=42).columns.tolist()
)

interaction_small = interaction_small[cols_to_keep]
'''

# Static Methods - DO NOT CHANGE

In [ ]:
# Construct full output paths
feature_plot_path = os.path.join(output_dir, 'feature_reduction.png')
predictions_path = os.path.join(output_dir, 'predictions.csv')
feature_importance_path = os.path.join(output_dir, 'feature_importances.csv')
model_path = os.path.join(output_dir, 'best_xgb_model.pkl')
bacteria_features_path = os.path.join(output_dir, 'feature_bacteria.txt')
phage_features_path = os.path.join(output_dir, 'feature_phage.txt')


# Model parameters
random_state = 100  # for reproducibility
cores = -1  # -1 uses all cores
ratio_test = 0.25  # Train/Test ratios
# RepeatedKFold parameters
n_splits = 10
n_repeats = 3

# REF
perc_step = 0.1 
n_features_first=2000
n_features_final = 45

set_config(transform_output="pandas")

gradient_boosting = GradientBoostingRegressor(n_estimators=1000, random_state=random_state)

cv = RepeatedKFold(n_splits=n_splits, n_repeats=n_repeats, random_state=random_state)

rfecv = RFECV(gradient_boosting,
            step=perc_step, 
            verbose=0,
            min_features_to_select=n_features_final, 
            cv=cv,
            n_jobs=cores, 
            scoring='neg_root_mean_squared_error')

rfe_init = RFE(estimator=gradient_boosting, 
        n_features_to_select=n_features_first,
        step=perc_step,
        verbose=0)

param_grid = {
    'n_estimators': [500, 750, 1000, 1250],  # Number of trees in the boosting ensemble
    'max_depth': [3, 5, 10, 20],  # Depth of the individual trees
    'learning_rate': [0.01, 0.05, 0.1, 0.2],  # Learning rate for boosting
    'subsample': [0.8, 0.9, 1.0],  # Fraction of samples used for fitting each tree
    'min_samples_split': [2, 5, 10],  # Minimum number of samples required to split a node
    'min_samples_leaf': [1, 2, 4]  # Minimum number of samples required in each leaf
}

tuning = RandomizedSearchCV(estimator=GradientBoostingRegressor(random_state=random_state), 
                                param_distributions=param_grid, 
                                cv=cv, 
                                scoring='neg_root_mean_squared_error',
                                verbose=3,
                                n_jobs=cores,
                                n_iter=25, 
                                refit=True,
                                random_state=random_state)

# PANPHAGE MODELS

In [ ]:
x = interaction_data.drop(columns=['Score'])
#x = interaction_small.drop(columns=['Score']) # Use downsampled data for testing
y = interaction_data['Score']
#y = interaction_small['Score'] #Use downsampled data for testing

# Stratify based on whether score is above or below 60
y_strat = (y >= 60).astype(int)
# ALD1 only has 1 positive interaction. We must label it as negative or train_test_split will raise exception
y_strat['CAN98_ALD1'] = 1
# For Panphage, we can further stratify by phage ID to ensure representation
indexes = interaction_data.index
phages = [idx.split('_')[1] for idx in indexes]
combined_strat = [f"{phage}_{y_strat}" for phage, y_strat in zip(phages, y_strat)]
# combined_strat

x_train, x_test, y_train, y_test = train_test_split(x, y, 
                                                    test_size=ratio_test, 
                                                    random_state=random_state,
                                                    stratify=combined_strat)

# Export test/train split:
# y_test = pd.DataFrame(y_test)
# y_test['dataset'] = 'test'
# y_train = pd.DataFrame(y_train)
# y_train['dataset'] = 'train'
# split = pd.concat([y_train, y_test])
# split.to_csv('test_train_split.csv')

# ---- 1. First RFE ----
rfe_stage1 = rfe_init.fit(x_train, y_train)
X_train_s1 = rfe_stage1.transform(x_train)
X_test_s1  = rfe_stage1.transform(x_test)

# ---- 2. Second RFE (RFECV) ----
rfe_stage2 = rfecv.fit(X_train_s1, y_train)
X_train_s2 = rfe_stage2.transform(X_train_s1)
X_test_s2  = rfe_stage2.transform(X_test_s1)

# ---- 3. Model tuning (GridSearchCV / RandomizedSearchCV / estimator) ----
tuning.fit(X_train_s2, y_train)
best_model = tuning.best_estimator_
best_model.feature_order_= best_model.feature_names_in_.tolist()

#Save the best model
with open(model_path, 'wb') as f:
        pickle.dump(best_model, f)

# ---- 4. Predict on test data ----
y_pred = best_model.predict(X_test_s2)
y_pred_train = best_model.predict(X_train_s2)

# Plotting feature reduction results with RMSE points
y_pred_rfe_init = rfe_init.predict(x_test)
rmse_rfe_init = np.sqrt(mean_squared_error(y_test, y_pred_rfe_init))

rmse_best = np.sqrt(mean_squared_error(y_train, y_pred_train))

# Get the number of features used by each model
n_features_rfe_init = n_features_first  # Initial RFE terminates at this number
n_features_best = X_train_s2.shape[1]   # Best model uses features after both RFE stages

plt.figure(figsize=(10, 6))
plt.plot(rfe_stage2.cv_results_['n_features'], rfe_stage2.cv_results_['mean_test_score'], marker='o', linestyle='-', color='b')
plt.title('Model Performance vs Number of Features')
plt.xlabel('Number of Features')
plt.ylabel('Cross-validation Score (Neg RMSE)')
plt.grid(True)
# Add RMSE as dots on the chart
plt.scatter(n_features_rfe_init, -rmse_rfe_init, color='red', s=100, zorder=5, label=f'Initial RFE RMSE: {rmse_rfe_init:.2f}')
plt.scatter(n_features_best, -rmse_best, color='green', s=100, zorder=5, label=f'Best Model RMSE: {rmse_best:.2f}')
plt.legend()
plt.savefig(feature_plot_path)
plt.close()

train_results = pd.DataFrame({
    'observed': y_train, 
    'predicted': y_pred_train, 
    'dataset':"train"
})  
test_results = pd.DataFrame({
    'observed': y_test, 
    'predicted': y_pred , 
    'dataset':"test"
}) 

combined_results = pd.concat([train_results, test_results])
combined_results.to_csv(predictions_path, index=True)

importance_df = pd.DataFrame({
    'Feature': best_model.feature_names_in_,
    'Importance': best_model.feature_importances_,
}) 
importance_df.to_csv(feature_importance_path, index=False)

# Filter features for bacteria and phage
features_bacteria = [feature.replace('_bacteria', '') for feature in best_model.feature_names_in_ if feature.endswith('_bacteria')]
features_phage = [feature.replace('_phage', '') for feature in best_model.feature_names_in_ if feature.endswith('_phage')]
print(f"Found {len(features_bacteria)} bacteria features and {len(features_phage)} phage features")

with open(bacteria_features_path, 'w') as file: 
    for feature in features_bacteria:
        file.write(feature + '\n')
with open(phage_features_path, 'w') as file: 
    for feature in features_phage:
        file.write(feature + '\n')

# Single Phage Models

In [ ]:
unique_phages = interaction_data.index.str.split('_').str[1].unique() 

# Dictionary to store individual phage models and results
phage_models = {}
phage_results = {}

# Same split as panphage model:
x = interaction_data.drop(columns=['Score'])
#x = interaction_small.drop(columns=['Score']) # Use downsampled data for testing
y = interaction_data['Score']
#y = interaction_small['Score'] #Use downsampled data for testing

# Stratify based on whether score is above or below 60
y_strat = (y >= 60).astype(int)
# ALD1 only has 1 positive interaction. We must label it as negative or train_test_split will raise exception
y_strat['CAN98_ALD1'] = 1
# For Panphage, we can further stratify by phage ID to ensure representation
indexes = interaction_data.index
phages = [idx.split('_')[1] for idx in indexes]
combined_strat = [f"{phage}_{y_strat}" for phage, y_strat in zip(phages, y_strat)]
# combined_strat

x_train, x_test, y_train, y_test = train_test_split(x, y, 
                                                    test_size=ratio_test, 
                                                    random_state=random_state,
                                                    stratify=combined_strat)

for phage in unique_phages:
    print(f"Training model for phage: {phage}")

    # Filter data for this specific phage
    phage_mask_train = x_train.index.str.endswith(f'_{phage}')
    phage_mask_test = x_test.index.str.endswith(f'_{phage}')

    x_train_phage = x_train[phage_mask_train]
    y_train_phage = y_train[phage_mask_train]
    x_test_phage = x_test[phage_mask_test]
    y_test_phage = y_test[phage_mask_test]

    # Create output paths for this phage
    phage_model_path = os.path.join(output_dir, "singlephage", f"phage_{phage}_model.pkl")
    phage_predictions_path = os.path.join(output_dir, "singlephage", f"phage_{phage}_predictions.csv")
    phage_feature_importance_path = os.path.join(output_dir, "singlephage", f"phage_{phage}_feature_importance.csv")
    phage_feature_plot_path = os.path.join(output_dir, "singlephage", f"phage_{phage}_feature_plot.png")
    phage_bacteria_features_path = os.path.join(output_dir, "singlephage", f"phage_{phage}_bacteria_features.txt")
    
    os.makedirs(os.path.dirname(phage_model_path), exist_ok=True)
    os.makedirs(os.path.dirname(phage_predictions_path), exist_ok=True)

    # ---- 1. First RFE ----
    rfe_stage1_phage = rfe_init.fit(x_train_phage, y_train_phage)
    X_train_s1_phage = rfe_stage1_phage.transform(x_train_phage)
    X_test_s1_phage = rfe_stage1_phage.transform(x_test_phage)
    
    # ---- 2. Second RFE (RFECV) ----
    rfe_stage2_phage = rfecv.fit(X_train_s1_phage, y_train_phage)
    X_train_s2_phage = rfe_stage2_phage.transform(X_train_s1_phage)
    X_test_s2_phage = rfe_stage2_phage.transform(X_test_s1_phage)
    
    # ---- 3. Model tuning ----
    tuning.fit(X_train_s2_phage, y_train_phage)
    best_model_phage = tuning.best_estimator_
    best_model_phage.feature_order_ = best_model_phage.feature_names_in_.tolist()
    
    # Save the phage-specific model
    with open(phage_model_path, 'wb') as f:
        pickle.dump(best_model_phage, f)
    
    # ---- 4. Predict on test data ----
    y_pred_phage = best_model_phage.predict(X_test_s2_phage)
    y_pred_train_phage = best_model_phage.predict(X_train_s2_phage)
    
    # Calculate RMSE
    rmse_test_phage = np.sqrt(mean_squared_error(y_test_phage, y_pred_phage))
    rmse_train_phage = np.sqrt(mean_squared_error(y_train_phage, y_pred_train_phage))
    
    print(f"  Phage {phage}: Train RMSE = {rmse_train_phage:.2f}, Test RMSE = {rmse_test_phage:.2f}")
    
    # Store model and results
    phage_models[phage] = {
        'model': best_model_phage,
        'train_rmse': rmse_train_phage,
        'test_rmse': rmse_test_phage,
        'n_samples': len(x_train_phage) + len(x_test_phage)
    }
    
    # Create results DataFrame
    train_results_phage = pd.DataFrame({
        'observed': y_train_phage, 
        'predicted': y_pred_train_phage, 
        'dataset': "train",
        'phage': phage
    })  
    test_results_phage = pd.DataFrame({
        'observed': y_test_phage, 
        'predicted': y_pred_phage, 
        'dataset': "test",
        'phage': phage
    }) 
    
    combined_results_phage = pd.concat([train_results_phage, test_results_phage])
    combined_results_phage.to_csv(phage_predictions_path, index=True)
    
    # Feature importance
    importance_df_phage = pd.DataFrame({
        'Feature': best_model_phage.feature_names_in_,
        'Importance': best_model_phage.feature_importances_,
    }) 
    importance_df_phage.to_csv(phage_feature_importance_path, index=False)
    
    # Filter features for bacteria and phage
    features_bacteria_phage = [feature.replace('_bacteria', '') for feature in best_model_phage.feature_names_in_ if feature.endswith('_bacteria')]
    
    with open(phage_bacteria_features_path, 'w') as file: 
        for feature in features_bacteria_phage:
            file.write(feature + '\n')
    print(f"  Completed phage {phage}: {len(features_bacteria_phage)} bacteria features")

# Optional: Create a summary of all phage models
summary_data = []
for phage, results in phage_models.items():
    summary_data.append({
        'phage': phage,
        'n_samples': results['n_samples'],
        'train_rmse': results['train_rmse'],
        'test_rmse': results['test_rmse']
    })

summary_df = pd.DataFrame(summary_data)
summary_df.to_csv(os.path.join(output_dir, "singlephage", "phage_models_summary.csv"), index=False)
print("Phage models summary saved to phage_models_summary.csv")